# R2 — MiniLM cosine feature

R2 เพิ่ม semantic similarity จาก `all-MiniLM-L6-v2` เป็น feature ที่ 18 แล้ว train IdentityMLP ใหม่ แต่ยังใช้ threshold มือเดิม.

In [ ]:
from pathlib import Path
import sys, json, inspect
import pandas as pd
from IPython.display import Markdown, display

def find_root():
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents,
                  Path(r'D:/66070260-Year3_Term2/Project1/Code')]
    for candidate in candidates:
        if (candidate / 'exp_lib.py').exists(): return candidate
    raise FileNotFoundError('Project root containing exp_lib.py was not found')

ROOT = find_root(); EXP = ROOT / 'experiments'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

def source(module, *names):
    for name in names:
        display(Markdown(f'### `{module.__name__}.{name}`'))
        print(inspect.getsource(getattr(module, name)))

def read_json(relative):
    return json.loads((ROOT / relative).read_text(encoding='utf-8'))

print('Project root:', ROOT)


## 1. Profile text → embedding → cosine

MiniLM encode `fullName + bio + location` ต่อ profile แล้ว cosine ของ profile pair ถูกเก็บเป็น `bert_cos`.

In [ ]:
import exp_r2_bert_feature as r2
source(r2, 'build_profile_texts', 'encode_profiles', 'compute_bert_cos')
bert = pd.read_parquet(EXP/'pair_bert_cos.parquet'); display(bert.head()); print(len(bert))

## 2. 18 features และ IdentityMLP

17 features เดิม + `bert_cos`; train, calibration และ test แยก role เพื่อกัน leakage.

In [ ]:
source(r2, 'IdentityMLP', 'train_mlp', 'train_r2_probabilities')

## 3. Manual decision

R2 ใช้ `MATCH=0.98`, `REVIEW=0.95`; R3 จะใช้ probability เดียวกัน แต่เปลี่ยน decision layer เป็น GA.

In [ ]:
source(r2, 'decide_manual', 'run_r2')
r = read_json('experiments/r2_results.json'); display(pd.DataFrame(r['splits']).T)